# JSON schema mode 2: a typed schema per application/action
Instead of free-form strings, encode every application, action and parameter in the schema itself.

![schema](json_schema_mode_2_schema.svg)

In [10]:
import json
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import JSON, Markdown, display

load_dotenv()  # loads GEMINI_API_KEY from .env
client = genai.Client()

from typing import Literal

from pydantic import BaseModel, Field

In [11]:
transcript = "Hey Google, navigate to Ara."

In [12]:
# --- maps ---
class Navigate(BaseModel):
    action: Literal["navigate"]
    destination: str
    mode: Literal["driving", "walking", "cycling", "transit"] | None = None


class SearchNearby(BaseModel):
    action: Literal["search_nearby"]
    query: str
    radius_km: float | None = None


class ShowTraffic(BaseModel):
    action: Literal["show_traffic"]


class Maps(BaseModel):
    """Navigation and location search"""

    application: Literal["maps"]
    command: Navigate | SearchNearby | ShowTraffic


# --- music ---
class Play(BaseModel):
    action: Literal["play"]
    song: str | None = None
    artist: str | None = None
    playlist: str | None = None


class Pause(BaseModel):
    action: Literal["pause"]


class Skip(BaseModel):
    action: Literal["skip"]


class SetVolume(BaseModel):
    action: Literal["set_volume"]
    level: int = Field(ge=0, le=100)


class Music(BaseModel):
    """Music streaming and playback"""

    application: Literal["music"]
    command: Play | Pause | Skip | SetVolume


# --- messages ---
class SendMessage(BaseModel):
    action: Literal["send_message"]
    recipient: str
    body: str


class ReadLatest(BaseModel):
    action: Literal["read_latest"]
    sender: str | None = None


class Messages(BaseModel):
    """Send and read text messages"""

    application: Literal["messages"]
    command: SendMessage | ReadLatest


# --- calendar ---
class CreateEvent(BaseModel):
    action: Literal["create_event"]
    title: str
    date: str = Field(description="YYYY-MM-DD")
    time: str | None = Field(default=None, description="HH:MM")
    duration_minutes: int | None = None


class ListEvents(BaseModel):
    action: Literal["list_events"]
    date: str = Field(description="YYYY-MM-DD")


class SetReminder(BaseModel):
    action: Literal["set_reminder"]
    text: str
    time: str = Field(description="HH:MM")


class Calendar(BaseModel):
    """Manage events and reminders"""

    application: Literal["calendar"]
    command: CreateEvent | ListEvents | SetReminder


# --- weather ---
class CurrentConditions(BaseModel):
    action: Literal["current_conditions"]
    location: str | None = None


class Forecast(BaseModel):
    action: Literal["forecast"]
    location: str | None = None
    days: int | None = None


class Weather(BaseModel):
    """Weather forecasts and conditions"""

    application: Literal["weather"]
    command: CurrentConditions | Forecast


class ParsedCommand(BaseModel):
    target: Maps | Music | Messages | Calendar | Weather

In [19]:
def const_to_enum(schema):
    """Pydantic renders single-value Literals as {"const": x}, which Gemini doesn't
    enforce. Rewrite them as {"enum": [x]} so the choice is actually constrained."""
    if isinstance(schema, dict):
        if "const" in schema:
            schema["enum"] = [schema.pop("const")]
        for value in schema.values():
            const_to_enum(value)
    elif isinstance(schema, list):
        for item in schema:
            const_to_enum(item)
    return schema


schema = const_to_enum(ParsedCommand.model_json_schema())
print(json.dumps(schema, indent=2))

{
  "$defs": {
    "Calendar": {
      "description": "Manage events and reminders",
      "properties": {
        "application": {
          "title": "Application",
          "type": "string",
          "enum": [
            "calendar"
          ]
        },
        "command": {
          "anyOf": [
            {
              "$ref": "#/$defs/CreateEvent"
            },
            {
              "$ref": "#/$defs/ListEvents"
            },
            {
              "$ref": "#/$defs/SetReminder"
            }
          ],
          "title": "Command"
        }
      },
      "required": [
        "application",
        "command"
      ],
      "title": "Calendar",
      "type": "object"
    },
    "CreateEvent": {
      "properties": {
        "action": {
          "title": "Action",
          "type": "string",
          "enum": [
            "create_event"
          ]
        },
        "title": {
          "title": "Title",
          "type": "string"
        },
        "date": {


## Input
No list of applications in the prompt any more: it all lives in the schema.

In [14]:
message = f"""For the following transcript, determine the target application the user
wants to use, the action to take, and any parameters for that action.

transcript: {transcript}"""
print(message)

For the following transcript, determine the target application the user
wants to use, the action to take, and any parameters for that action.

transcript: Hey Google, navigate to Ara.


## Model call

In [20]:
start = time.perf_counter()
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=message,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_json_schema=schema,
    ),
)
print(f"Client call took {time.perf_counter() - start:.2f}s")

Client call took 1.48s


## Output

In [22]:
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""{
  "target": {
    "application": "maps",
    "command": {
      "action": "navigate",
      "destination": "Ara",
      "mode": null
    }
  }
}""",
        thought_signature=b'\x12`\n^\x01i\x14}\x13\xf4)\x8e\xb0\x96p\xbd\x96\xb9\xe0r\xe2\xc3\xfb\xd7\xec\xf3x\x95\xd0c\xa8B\x04\xdc/E\x0fy\xb0q\x88\x13\xb8\xe8.\xb5RD\xd23\tc\xf8]\x17\xaa-s\xd9\xfa\xa7\xaa\x88\xad|\xdfK\xc8GK\x8b\xdb\xb0\x07\xfd\x8e\xa94\xf6\x9a\x1d\xab\x92u\xb6\xee\xf2\xb8}K<k1\xeeSpH\xaf'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='Wcv3auKTFtrdg8UPgYqF2Q0' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=51,
  prompt_token_count=40,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TE

In [23]:
result = ParsedCommand.model_validate(response.parsed)
result

ParsedCommand(target=Maps(application='maps', command=Navigate(action='navigate', destination='Ara', mode=None)))

In [24]:
print(result.model_dump_json(indent=2, exclude_none=True))

{
  "target": {
    "application": "maps",
    "command": {
      "action": "navigate",
      "destination": "Ara"
    }
  }
}
